### PREPROCESS ###

### Load Data ###

In [6]:
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from sklearn.metrics import classification_report
from tqdm import tqdm

In [7]:
MODEL_NAME = "indobenchmark/indobert-base-p1"
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [9]:
train_df = pd.read_csv("../data_labelling/train_labeled.csv")
val_df   = pd.read_csv("../data_labelling/val_labeled.csv")
test_df  = pd.read_csv("../data_labelling/test_labeled.csv")

train_df["label"] = train_df["label"].astype(int)
val_df["label"]   = val_df["label"].astype(int)
test_df["label"]  = test_df["label"].astype(int)

In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_texts(texts, tokenizer, max_len=128):
    return tokenizer(
        texts.tolist(),
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )

In [11]:
class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.values

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [14]:
from transformers import BertForSequenceClassification, AutoTokenizer

teacher_model = BertForSequenceClassification.from_pretrained("./model1_manual")
teacher_tokenizer = AutoTokenizer.from_pretrained("./model1_manual")

teacher_model.to(device)
teacher_model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(50000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [15]:
train_encodings = tokenize_texts(train_df["cleaned_text"], tokenizer)
train_loader = DataLoader(
    SentimentDataset(train_encodings, train_df["label"]),
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [16]:
pseudo_labels = []
confidences = []

with torch.no_grad():
    for batch in tqdm(train_loader, desc="Generating Pseudo Labels"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = teacher_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probs = F.softmax(outputs.logits, dim=1)
        conf, preds = torch.max(probs, dim=1)

        pseudo_labels.extend(preds.cpu().numpy())
        confidences.extend(conf.cpu().numpy())

Generating Pseudo Labels: 100%|██████████| 69/69 [00:18<00:00,  3.77it/s]


In [20]:
train_pseudo_df = train_df.copy()
train_pseudo_df["label"] = pseudo_labels
train_pseudo_df["confidence"] = confidences

In [21]:
CONF_THRESHOLD = None

if CONF_THRESHOLD is not None:
    train_pseudo_df = train_pseudo_df[
        train_pseudo_df["confidence"] >= CONF_THRESHOLD
    ].reset_index(drop=True)

In [22]:
train_pseudo_enc = tokenize_texts(train_pseudo_df["cleaned_text"], tokenizer)
val_encodings    = tokenize_texts(val_df["cleaned_text"], tokenizer)
test_encodings   = tokenize_texts(test_df["cleaned_text"], tokenizer)

train_dataset = SentimentDataset(train_pseudo_enc, train_pseudo_df["label"])
val_dataset   = SentimentDataset(val_encodings, val_df["label"])
test_dataset  = SentimentDataset(test_encodings, test_df["label"])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [23]:
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)
model.to(device)

optimizer = AdamW(model.parameters(), lr=LR)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [24]:
def train_epoch(model, loader):
    model.train()
    total_loss = 0

    for batch in tqdm(loader, desc="Training"):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [25]:
def eval_epoch(model, loader):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            total_loss += outputs.loss.item()

    return total_loss / len(loader)

In [26]:
for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader)
    val_loss   = eval_epoch(model, val_loader)

    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss  : {val_loss:.4f}")
    print("-" * 30)

Training: 100%|██████████| 69/69 [00:47<00:00,  1.46it/s]


Epoch 1
Train Loss: 0.8890
Val Loss  : 0.7422
------------------------------


Training: 100%|██████████| 69/69 [00:47<00:00,  1.46it/s]


Epoch 2
Train Loss: 0.5156
Val Loss  : 0.7437
------------------------------


Training: 100%|██████████| 69/69 [00:47<00:00,  1.46it/s]


Epoch 3
Train Loss: 0.2029
Val Loss  : 1.0937
------------------------------


In [27]:
def get_predictions(model, loader):
    model.eval()
    preds, labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            pred = torch.argmax(outputs.logits, dim=1)

            preds.extend(pred.cpu().numpy())
            labels.extend(batch["labels"].numpy())

    return preds, labels

In [ ]:
y_pred, y_true = get_predictions(model, test_loader)

print(classification_report(
    y_true,
    y_pred,

    
    target_names=["Negatif", "Netral", "Positif"]
))

              precision    recall  f1-score   support

     Negatif       0.81      0.71      0.75       130
      Netral       0.58      0.50      0.54        60
     Positif       0.54      0.75      0.62        60

    accuracy                           0.67       250
   macro avg       0.64      0.65      0.64       250
weighted avg       0.69      0.67      0.67       250

